# Lesson 3 | Why is “we both use LIF” not enough?

In the first two lessons we built a runnable **Leaky Integrate-and-Fire (LIF)** neuron and saw that finite-width representation can change its trajectory.

Now a very engineering-like, but also very scientific, question appears:

> **If Python, AI, and future hardware interpret “the same LIF” differently, which one is correct?**

The primary new concept in this lesson is: **freeze testable model semantics before implementation.**

## 1. Three words first: specification, semantics, test oracle

### Specification
A **specification**, often shortened to **spec**, is the explicit rule set that says how the system must behave. It is not the code itself; it is the contract the code must satisfy.

### Semantics
**Semantics** means what a rule actually means. “Fire when threshold is reached” sounds clear, but does that mean `V > threshold` or `V >= threshold`? That difference is semantic.

### Test oracle
A **test oracle** is the authoritative basis for deciding whether a test result is correct.

If Python and hardware disagree, we cannot simply say “Python is always right” or “hardware is always right.” The authority should be the frozen specification and the reference tests derived from it.

## 2. One character can define two different models

Suppose the updated membrane state is exactly equal to threshold:

- Rule A: spike when `new_v >= threshold`;
- Rule B: spike when `new_v > threshold`.

The code differs by one character, but the behavior differs.

Let us deliberately construct an example that lands exactly on the boundary.

In [ ]:
def step_ge(v, current, alpha=1.0, threshold=1.0, reset=0.0):
    new_v = alpha * v + current
    spike = new_v >= threshold
    return (reset if spike else new_v), spike

def step_gt(v, current, alpha=1.0, threshold=1.0, reset=0.0):
    new_v = alpha * v + current
    spike = new_v > threshold
    return (reset if spike else new_v), spike

v0 = 0.75
current = 0.25

print('>= rule:', step_ge(v0, current))
print(' > rule:', step_gt(v0, current))

## 3. Observe: why can this be more useful than many random tests?

This is a **boundary test**: we deliberately choose an input where two definitions are most likely to diverge.

With ordinary random inputs, `>` and `>=` might behave identically for a long time. That does not mean they have the same semantics.

A key testing intuition is:

> Good tests are not only numerous; they actively seek inputs that distinguish competing interpretations.

## 4. Another ambiguity: leak then input, or input then leak?

Consider two update rules:

A. `new_v = alpha * v + input`

B. `new_v = alpha * (v + input)`

Both might casually be described as “leaky integration,” but they are not numerically identical.

In [ ]:
v = 0.8
current = 0.4
alpha = 0.9

rule_a = alpha * v + current
rule_b = alpha * (v + current)

print('A: leak old state, then add input ->', rule_a)
print('B: add input, then leak everything  ->', rule_b)

## 5. “A common LIF implementation” is not a specification

Papers, textbooks, and online examples use many LIF variants:

- continuous-time differential equations or discrete-time updates;
- `>` or `>=` for threshold;
- immediate reset or next-step reset;
- refractory periods that ignore input or continue accumulating it;
- optional noise or tonic drive.

Many of these are reasonable for different purposes.

So an engineering project cannot stop at “we use standard LIF.” We must say **which LIF semantics this project uses.**

## 6. Which semantics must we freeze?

At minimum, answer these questions:

| Question | Why it matters |
|---|---|
| When is state `V` read? | Determines whether the update uses old or already-updated state |
| What is the order of leak and input? | Different orders can produce different values |
| Does threshold use `>` or `>=`? | Boundary behavior changes |
| What value is used after reset? | Defines the next starting state |
| How long is the refractory period? | Determines when firing is allowed again |
| Is input accepted during refractory? | Determines whether input is lost or accumulated |
| Where does rounding happen? | Changes fixed-point trajectories |
| How is overflow handled? | Saturation and wraparound differ drastically |
| How are multiple same-step inputs combined? | Defines accumulation and concurrency semantics |

This table eventually becomes part of the model contract.

## 7. What does “freeze” mean?

Freezing does not mean “never change this again.” It means:

1. choose one rule for the current version;
2. give that version a traceable definition;
3. write tests that distinguish critical semantics;
4. make Python, future hardware, and tests all refer to the same definition;
5. if the rule changes later, treat that as a version change rather than a silent code edit.

This lets us answer an important question:

> **Why is this implementation correct? Because it matches an explicit, tested specification.**

In [ ]:
semantic_decisions = {
    'threshold_comparison': 'TBD: >= or >',
    'update_order': 'TBD: leak_then_input or input_then_leak',
    'reset_value': 'TBD',
    'refractory_steps': 'TBD',
    'input_during_refractory': 'TBD',
    'rounding_rule': 'TBD',
    'overflow_rule': 'TBD',
    'same_step_input_accumulation': 'TBD',
}

for key, value in semantic_decisions.items():
    print(f'{key:30s} {value}')

## 8. Try It: design a minimal counterexample for each ambiguity

Do not rush to fill every `TBD`. Choose two questions and design an input that forces two candidate rules to produce different outputs.

Examples:

- `>` versus `>=`: make candidate voltage exactly equal threshold;
- saturation versus wraparound: intentionally exceed the maximum representable value;
- input during refractory: inject a large input during refractory and inspect the state after recovery.

If you cannot construct a test that distinguishes two rules, you may not yet understand the difference between them.

## 9. AI Task

Ask AI to draft a checklist of ambiguities that must be resolved before implementing LIF. Require each item to include:

1. two candidate semantics;
2. a minimal distinguishing test;
3. why the choice could affect spike sequence.

Then a human chooses which semantics enter the v0 specification.

AI can help us discover missing questions, but “this is the common implementation” is not enough reason for an undocumented decision.

## 10. Human Check

Without AI, explain:

- specification, semantics, and test oracle;
- why `>` versus `>=` is not merely code style;
- why passing many random tests does not eliminate a boundary-semantics bug;
- why the Python reference is also constrained by the spec rather than automatically being the highest authority;
- what the correct process is if we later want a different LIF semantics.

## 11. Engineering Handoff

The output of this lesson is not more algorithm code. It is a **spec checkpoint**:

- write v0 neuron semantics into MDD;
- write boundary tests and oracles into TDD;
- record the requirement/design/test/build relationships in TRACE.

Later we will use **Register-Transfer Level (RTL)** to describe digital hardware: what state registers hold and how data is transformed and transferred across clock cycles. Today you only need to recognize the term; formal RTL comes later.

## 12. Project Trace

- Lesson ID: `LSN-003`
- Engineering slice: `RMD-003`
- Process requirements: `PFR3 / PFR4`
- Process design: `PDP3 / PDP4`

These IDs exist for long-term traceability, not memorization.

## 13. Exit Ticket

Before continuing:

1. explain specification, semantics, and test oracle;
2. give at least two examples of models that are both called LIF but have different semantics;
3. make sure implementation-relevant choices in `semantic_decisions` are not unconscious defaults;
4. attach at least one deliberately failing test to each critical decision.

The next lesson moves us closer to digital hardware:

> A software variable naturally remembers `v`; how can a circuit remember the previous `v`?